## 📥 FASE 1: EXTRACT — Carga y diagnóstico de datos

In [ ]:
import pandas as pd
import glob
import os

# DATA FRAMES
df_restaurantes = pd.read_csv('data/restaurants.csv')

ruta_menus = 'data/restaurant-menus.csv'
archivos_menu = sorted(glob.glob(os.path.join(ruta_menus, '*.csv')))
print(f"Archivos de menú encontrados: {len(archivos_menu)}")
for f in archivos_menu:
    print(f)

In [ ]:
# CONSOLIDACIÓN DE MENUS EN UNO SOLO
df_menus = pd.concat(
    [pd.read_csv(f) for f in archivos_menu],
    ignore_index=True
)
print(f"Total filas df_menus: {df_menus.shape[0]:,}")

In [ ]:
# --- Diagnóstico de RESTAURANTES ---
print("=== RESTAURANTES ===")
print("Dimensiones:", df_restaurantes.shape)
print("\nColumnas:", df_restaurantes.columns.tolist())
print("\nTipos de datos:\n", df_restaurantes.dtypes)
print("\nValores nulos:\n", df_restaurantes.isnull().sum())

In [ ]:
# --- Diagnóstico de MENÚS ---
print("=== MENÚS ===")
print("Dimensiones:", df_menus.shape)
print("\nColumnas:", df_menus.columns.tolist())
print("\nTipos de datos:\n", df_menus.dtypes)
print("\nValores nulos:\n", df_menus.isnull().sum())

## 🧹 FASE 2: TRANSFORM — Limpieza e ingeniería de características

In [ ]:
# MAPEO DE PRECIOS
mapa_precios = {
    '$':    'Económico',
    '$$':   'Moderadamente caro',
    '$$$':  'Caro',
    '$$$$': 'Muy caro'
}
df_restaurantes['price_range'] = df_restaurantes['price_range'].map(mapa_precios)
print(df_restaurantes['price_range'].value_counts())

In [ ]:
# DESANIDADO DE DIRECCIÓN
df_restaurantes[['calle', 'ciudad', 'estado_zip']] = (
    df_restaurantes['full_address']
    .str.split(',', expand=True, n=2)
)
df_restaurantes['ciudad'] = df_restaurantes['ciudad'].str.strip()
print(df_restaurantes[['calle', 'ciudad', 'estado_zip']].head())

In [ ]:
# FILTRO DE CALIDAD
df_restaurantes = df_restaurantes[
    df_restaurantes['score'].notnull() & (df_restaurantes['score'] != 0)
]
df_restaurantes['category'] = df_restaurantes['category'].str.lower()
print(f"Restaurantes después del filtro de calidad: {df_restaurantes.shape[0]:,}")

In [ ]:
# LIMPIEZA DE PRECIOS EN MENÚS
def limpiar_precio(valor):
    if pd.isnull(valor):
        return None
    valor = str(valor).replace(' USD', '').replace(',', '.')
    try:
        return float(valor)
    except ValueError:
        return None

df_menus['price'] = df_menus['price'].apply(limpiar_precio)

In [ ]:
# ELIMINAR ÍTEMS CON PRECIO NULO O CERO
df_menus = df_menus[df_menus['price'].notnull() & (df_menus['price'] != 0.0)]
print(f"Ítems de menú después de la limpieza: {df_menus.shape[0]:,}")

## 🔗 Optimización de memoria y cruce de datos (Merge)

In [ ]:
# FILTRO: solo restaurantes con más de 100 calificaciones
df_restaurantes_filtrado = df_restaurantes[df_restaurantes['ratings'] > 100]
print(f"Restaurantes consolidados (más de 100 ratings): {df_restaurantes_filtrado.shape[0]:,}")

In [ ]:
# INNER JOIN entre restaurantes filtrados y menús
df_master = pd.merge(
    df_restaurantes_filtrado,
    df_menus,
    left_on='id',
    right_on='restaurant_id',
    how='inner'
)
print(f"Dimensiones del DataFrame master: {df_master.shape}")
print(df_master.head())

## 💾 FASE 3: LOAD — Carga a base de datos SQLite

In [ ]:
from sqlalchemy import create_engine, text

engine = create_engine('sqlite:///delivery_insights.db')

In [ ]:
df_master.to_sql('master_food_data', engine, if_exists='replace', index=False)
print("✅ Carga completada en delivery_insights.db")

## 📊 FASE 4: BUSINESS INTELLIGENCE — Reporte al CEO

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.backends.backend_pdf import PdfPages
import seaborn as sns

conn = engine.connect()

In [ ]:
# --- EJECUTAR LAS 4 QUERIES ---

# Query 1: Top 5 ciudades
query1 = """
SELECT ciudad, COUNT(DISTINCT id) AS cantidad_restaurantes
FROM master_food_data
GROUP BY ciudad
ORDER BY cantidad_restaurantes DESC
LIMIT 5
"""
df_q1 = pd.read_sql(text(query1), conn)
print("Q1 ✅"); print(df_q1)

# Query 2: Distribución de rango de precios
query2 = """
SELECT price_range, COUNT(DISTINCT id) AS cantidad
FROM master_food_data
GROUP BY price_range
"""
df_q2 = pd.read_sql(text(query2), conn)
print("Q2 ✅"); print(df_q2)

# Query 3: Precio promedio en top 3 ciudades
top3_ciudades = tuple(df_q1['ciudad'].head(3))
query3 = f"""
SELECT ciudad, AVG(price) AS precio_promedio
FROM master_food_data
WHERE ciudad IN {top3_ciudades}
GROUP BY ciudad
ORDER BY precio_promedio DESC
"""
df_q3 = pd.read_sql(text(query3), conn)
print("Q3 ✅"); print(df_q3)

# Query 4: Correlación precio-calidad
query4 = """
SELECT price_range, AVG(score) AS score_promedio
FROM master_food_data
GROUP BY price_range
"""
df_q4 = pd.read_sql(text(query4), conn)
orden = ['Económico', 'Moderadamente caro', 'Caro', 'Muy caro']
df_q4['price_range'] = pd.Categorical(df_q4['price_range'], categories=orden, ordered=True)
df_q4 = df_q4.sort_values('price_range')
print("Q4 ✅"); print(df_q4)

In [ ]:
# --- PÁGINA 1: FIGURA CON LOS 4 GRÁFICOS ---
fig1, axes = plt.subplots(2, 2, figsize=(18, 12))
fig1.suptitle(
    'Reporte de Inteligencia de Mercado — Food Delivery',
    fontsize=18,
    fontweight='bold',
    y=1.01
)

# Gráfico 1: Top 5 ciudades
sns.barplot(
    ax=axes[0, 0],
    data=df_q1,
    x='cantidad_restaurantes',
    y='ciudad',
    hue='ciudad',
    orient='h',
    palette='viridis',
    legend=False
)
axes[0, 0].set_title('1. Top 5 ciudades con más restaurantes', fontsize=13, fontweight='bold')
axes[0, 0].set_xlabel('Cantidad de restaurantes')
axes[0, 0].set_ylabel('Ciudad')

# Gráfico 2: Distribución de precios
axes[0, 1].pie(
    df_q2['cantidad'],
    labels=df_q2['price_range'],
    autopct='%1.1f%%',
    startangle=90,
    colors=sns.color_palette('Set2', len(df_q2))
)
axes[0, 1].set_title('2. Distribución por rango de precios', fontsize=13, fontweight='bold')

# Gráfico 3: Precio promedio top 3 ciudades
sns.barplot(
    ax=axes[1, 0],
    data=df_q3,
    x='ciudad',
    y='precio_promedio',
    hue='ciudad',
    palette='magma',
    legend=False
)
axes[1, 0].set_title('3. Precio promedio de menú (top 3 ciudades)', fontsize=13, fontweight='bold')
axes[1, 0].set_ylabel('Precio promedio ($)')
axes[1, 0].set_xlabel('Ciudad')

# Gráfico 4: Correlación precio-calidad
sns.lineplot(
    ax=axes[1, 1],
    data=df_q4,
    x='price_range',
    y='score_promedio',
    marker='o',
    color='steelblue',
    linewidth=2.5
)
axes[1, 1].set_title('4. ¿El precio garantiza mejor calidad?', fontsize=13, fontweight='bold')
axes[1, 1].set_ylabel('Score promedio')
axes[1, 1].set_xlabel('Rango de precios')
axes[1, 1].tick_params(axis='x', rotation=15)

plt.tight_layout()

In [ ]:
# --- PÁGINA 2: CONCLUSIONES ---
fig2, ax = plt.subplots(figsize=(18, 12))
ax.axis('off')  # Oculta los ejes, solo queremos texto

conclusiones = """
CONCLUSIONES PARA EL DIRECTORIO
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━


📍  Gráfico 1 — Penetración Geográfica

    Milwaukee y Seattle concentran casi el 80% de los restaurantes consolidados, siendo las
    únicas ciudades con más de 100 establecimientos activos. El mercado está altamente
    concentrado en dos plazas clave.

    → Recomendación: La startup debería lanzar primero en Milwaukee y Seattle,
      donde existe mayor demanda comprobada y densidad competitiva.


💰  Gráfico 2 — Distribución de Precios

    El 67.1% de los restaurantes son económicos y el 23.4% moderadamente caros.
    La hipótesis del CEO de que "todos los competidores son caros" queda refutada:
    menos del 10% del mercado opera en el segmento caro.

    → Recomendación: Competir en precio bajo es el segmento más saturado.
      Posicionarse como "moderado con buena calidad" puede ser más diferenciador.


🍽️  Gráfico 3 — Estrategia de Menú por Ciudad

    Seattle tiene el precio promedio más alto por plato (~$12), seguida por
    Lynnwood (~$11) y Milwaukee (~$9.5). A pesar de tener más restaurantes,
    Milwaukee opera con márgenes por plato más bajos.

    → Recomendación: Si la estrategia es posicionarse como premium, Seattle
      ofrece mayor tolerancia al precio por parte del consumidor.


⭐  Gráfico 4 — Correlación Precio-Calidad

    Los restaurantes económicos y moderadamente caros tienen scores casi idénticos
    (~4.63), mientras que los caros caen notoriamente a ~4.43. Pagar más no garantiza
    mejor experiencia para el usuario.

    → Recomendación: La startup puede entrar al mercado con precios competitivos
      sin sacrificar calidad percibida. Esta es la mayor oportunidad identificada.


━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

CONCLUSIÓN GENERAL

El mercado de food delivery en EE.UU. está concentrado en Milwaukee y Seattle, dominado
por opciones económicas y con una anomalía clave: los restaurantes más caros no son los
mejor valorados. La startup debería apuntar a Milwaukee como primer mercado por volumen,
con una propuesta de valor económica o moderada que iguale la calidad de los líderes actuales.
"""

ax.text(
    0.05, 0.97,
    conclusiones,
    transform=ax.transAxes,
    fontsize=11,
    verticalalignment='top',
    fontfamily='monospace',
    bbox=dict(boxstyle='round', facecolor='#f5f5f5', alpha=0.8)
)

fig2.suptitle(
    'Conclusiones y Recomendaciones Estratégicas',
    fontsize=16,
    fontweight='bold'
)

plt.tight_layout()

In [ ]:
# --- EXPORTAR PDF DE 2 PÁGINAS ---
pdf_path = 'reporte_CEO.pdf'

with PdfPages(pdf_path) as pdf:
    pdf.savefig(fig1, bbox_inches='tight', dpi=150)   # Página 1: gráficos
    pdf.savefig(fig2, bbox_inches='tight', dpi=150)   # Página 2: conclusiones

print(f"✅ Reporte exportado como '{pdf_path}' (2 páginas)")

plt.show()

### Conclusiones por gráfico

**Gráfico 1:** Milwaukee y Seattle concentran casi el 80% del mercado — son las plazas prioritarias de lanzamiento.

**Gráfico 2:** El 67.1% del mercado es económico, refutando la hipótesis del CEO. El segmento moderado es la oportunidad más diferenciadora.

**Gráfico 3:** Seattle es la ciudad con mayor precio promedio por plato (~$12), lo que indica mayor tolerancia al precio por parte del consumidor.

**Gráfico 4:** Los restaurantes más caros tienen peor puntaje que los económicos — el precio no garantiza calidad, lo que abre una oportunidad clara para la startup.